source ~/miniconda3/etc/profile.d/conda.sh && conda activate moai_project


nlp/sentiment.py - how it works

1. truncation=True — automatically cuts articles that are too long for BERT's 512-token limit, so it won't crash on long texts
2. self.pipe(text) i analyze the text of the article
3. analyze_batch(texts) - jumps over multiple texts and returns a table
4. aggregate_daily(df, date_col) - 
            avg_sentiment — mean of all article scores that day
            sentiment_std — how much articles disagreed that day (high std = mixed news)
            num_articles — article count per day (useful for the "volume spike" contrarian signal)
5. if __name__ == "__main__": - Runs when execute python sentiment.py directly. Tests on 5 sample headlines and prints results

Then you can test the sentiment pipeline:


cd src && python -m nlp.sentiment

---
# NewsAPI - collecting articles for sentiment analysis

**What is it?** NewsAPI (newsapi.org) is a REST API to search millions of news articles from 150,000+ sources.

**Why we use it:** We need real financial articles (titles + descriptions) to run FinBERT on. NewsAPI gives us structured data with dates, which we need for daily sentiment aggregation.

**Free tier limits:**
- 100 requests/day, 100 articles per request
- Articles go back 1 month
- Requires API key (get one at newsapi.org/register)

**What it returns per article:**
- `title` — headline (what we feed to FinBERT)
- `description` — short summary (can also be analyzed)
- `publishedAt` — date (needed for daily aggregation)
- `source` — which outlet published it
- `url` — link to full article

## Setup
Install: `pip install newsapi-python`
Get API key at newsapi.org/register

## How collect_news.py works
- Uses NewsApiClient to fetch articles by keyword (e.g. "SPY OR S&P 500")
- API key loaded from .env (gitignored)
- Saves title, description, date, source, url to data/raw/spy_newsapi_articles.csv
- Run: `python -m src.data.collect_news`

## Cleaning articles (clean_articles.py)
Filters out junk from raw NewsAPI data — only keeps financially relevant, English, non-duplicate articles.
Drops: empty titles, duplicates, non-English, irrelevant (no finance keywords in title/description).
100 raw → ~30 clean articles.
- Run: `python -m src.data.clean_articles`
- Input: data/raw/spy_newsapi_articles.csv
- Output: data/processed/spy_articles_clean.csv